# 🚁 Notebook 10 — Final Drone Project

### Everything you've learned, in one flight simulator

Congratulations — you've come a long way. Look at the journey:

1. **Dynamic systems** — position, velocity, acceleration, and simulating motion step by step.
2. **Forces & gravity** — Newton's `a = F/m`, thrust vs gravity.
3. **Open vs closed loop** — sensing, error, and feedback.
4. **P** — proportional control and the stubborn steady-state error.
5. **I** — integral control erases it (and windup / anti-windup).
6. **D** — derivative damping (and taming noise with a filter).
7. **PID** — the three terms cooperating, with a live dashboard.
8. **Tuning** — a repeatable workflow, sweeps, and metrics.
9. **Practical issues** — noise, saturation, wind, delay, and robustness.

This final notebook brings it all together into a **polished, self-contained drone simulator**: a
professional dashboard, an animated drone, automatic performance metrics, a full interactive
cockpit, and a set of **challenge problems** to prove your skills. Everything here runs on its own —
no earlier notebook required.

---

## 🎯 Learning objectives
- Assemble a clean, **self-contained** PID drone simulator.
- Read a **professional multi-panel dashboard** and an automatic metrics report.
- Solve **challenge problems**: setpoint tracking, a payload drop, gust rejection, minimal-overshoot tuning.

## 🧩 Prerequisites
Notebooks 01–09 (ideally), but this notebook stands alone.

## ⏱️ Estimated time
About **60–90 minutes** (including the challenges).

## 🧠 Concepts you know so far
Everything: dynamics, forces, feedback, P/I/D, tuning, robustness.

## 🔜 What you'll do here
Put it all to work, then take on open-ended flight challenges. 🎓


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


## 1. The complete simulator, in one place

Below is everything you built, gathered into one tidy toolkit: a physics **engine**, the
**PIDController**, and a **metrics** helper. The engine is slightly upgraded so `target`, `mass`,
and `wind` can each be either a fixed number **or a function of time** — that's what lets us do the
challenges (a mid-flight setpoint change, a payload drop, a sudden gust). Read the comments; it's
all familiar.


In [ ]:
from collections import deque

def _val(x, t):
    """Allow a parameter to be a constant OR a function of time t."""
    return x(t) if callable(x) else x

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=16.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone. target/mass/wind may be constants or functions of time."""
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)
    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "target", "p", "i", "d")
    hist = {k: [] for k in keys}
    for _ in range(int(total_time / dt)):
        tgt = _val(target, t); m = _val(mass, t); w = _val(wind, t)
        buf.append(x)
        measured = buf[0] + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = tgt - measured
        thrust = controller(tgt, measured, dt)
        thrust = min(max(thrust, min_thrust), max_thrust)
        a = (thrust - m * g + w) / m
        terms = getattr(controller, "last_terms", {"p": 0, "i": 0, "d": 0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error, tgt,
                                 terms.get("p", 0), terms.get("i", 0), terms.get("d", 0))):
            hist[k].append(val)
        v += a * dt; x += v * dt
        if x < 0: x, v = 0.0, 0.0
        t += dt
    return {k: np.array(val) for k, val in hist.items()}


class PIDController:
    """Complete PID: anti-windup + filtered derivative on measurement (no gravity feed-forward)."""
    def __init__(self, Kp=0.0, Ki=0.0, Kd=0.0, i_limit=15.0, d_filter=0.1, use_anti_windup=True):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.i_limit, self.d_filter, self.use_anti_windup = i_limit, d_filter, use_anti_windup
        self.integral = 0.0; self.prev_measured = None; self.d_state = 0.0
        self.last_terms = {"p": 0.0, "i": 0.0, "d": 0.0}
    def __call__(self, target, measured, dt):
        error = target - measured
        p = self.Kp * error
        self.integral += error * dt
        if self.use_anti_windup and self.Ki > 0:
            cap = self.i_limit / self.Ki
            self.integral = max(-cap, min(cap, self.integral))
        i = self.Ki * self.integral
        raw = 0.0 if self.prev_measured is None else (measured - self.prev_measured) / dt
        self.prev_measured = measured
        self.d_state += self.d_filter * (raw - self.d_state)
        d = -self.Kd * self.d_state
        self.last_terms = {"p": p, "i": i, "d": d}
        return p + i + d


def performance_metrics(t, x, target, band=0.05):
    t = np.asarray(t, float); x = np.asarray(x, float)
    final = float(np.mean(x[int(0.9 * len(x)):])); sse = target - final
    idx = np.where(x >= 0.9 * target)[0]
    rise = float(t[idx[0]]) if len(idx) else float("nan")
    overshoot = max(0.0, (float(np.max(x)) - target) / target * 100.0)
    outside = np.where(np.abs(x - target) > band * target)[0]
    settle = float(t[outside[-1]]) if len(outside) else 0.0
    return dict(rise_time=rise, overshoot_pct=overshoot, settling_time=settle,
                steady_state_error=sse, final_value=final)

print("Toolkit ready: simulate(), PIDController, performance_metrics().")


## 2. A professional dashboard

One function that turns a simulation into a six-panel readout: altitude vs target, velocity,
acceleration, thrust (with the saturation limit drawn in), tracking error, and the P/I/D
contributions. We'll reuse it everywhere below.


In [ ]:
def dashboard(res, title="PID flight dashboard", max_thrust=None):
    t = res["t"]
    fig, ax = plt.subplots(2, 3, figsize=(15, 7)); fig.suptitle(title, fontsize=14, fontweight="bold")
    ax[0,0].plot(t, res["x"], color="tab:blue", lw=2, label="altitude")
    ax[0,0].plot(t, res["target"], color="seagreen", ls="--", label="target")
    ax[0,0].set_title("Altitude vs target"); ax[0,0].set_ylabel("m"); ax[0,0].legend(fontsize=8)
    ax[0,1].plot(t, res["v"], color="tab:green", lw=1.8); ax[0,1].axhline(0, color="gray", ls=":")
    ax[0,1].set_title("Velocity"); ax[0,1].set_ylabel("m/s")
    ax[0,2].plot(t, res["a"], color="tab:olive", lw=1.5); ax[0,2].axhline(0, color="gray", ls=":")
    ax[0,2].set_title("Acceleration"); ax[0,2].set_ylabel("m/s^2")
    ax[1,0].plot(t, res["thrust"], color="tab:purple", lw=1.8)
    if max_thrust: ax[1,0].axhline(max_thrust, color="tab:red", ls=":", label="max thrust"); ax[1,0].legend(fontsize=8)
    ax[1,0].set_title("Thrust command"); ax[1,0].set_ylabel("N")
    ax[1,1].plot(t, res["error"], color="tab:red", lw=1.5); ax[1,1].axhline(0, color="gray", ls=":")
    ax[1,1].set_title("Tracking error"); ax[1,1].set_ylabel("m")
    ax[1,2].plot(t, res["p"], label="P", color="tab:orange")
    ax[1,2].plot(t, res["i"], label="I", color="tab:red")
    ax[1,2].plot(t, res["d"], label="D", color="tab:blue")
    ax[1,2].set_title("P / I / D contributions"); ax[1,2].set_ylabel("N"); ax[1,2].legend(fontsize=8)
    for a in ax.flat: a.set_xlabel("time (s)"); a.grid(True, alpha=0.3)
    plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

# Flagship demo: a solid all-round tuning in mildly realistic conditions
demo = simulate(PIDController(Kp=4.0, Ki=1.2, Kd=4.0, d_filter=0.1),
                total_time=16.0, noise_std=0.02, max_thrust=25.0)
dashboard(demo, "Flagship demo: well-tuned PID (light noise)", max_thrust=25.0)


## 3. Automatic performance report


In [ ]:
def report(res, target=10.0):
    m = performance_metrics(res["t"], res["x"], target)
    print("="*44); print("   FLIGHT PERFORMANCE REPORT"); print("="*44)
    print(f"  Rise time (to 90%) : {m['rise_time']:.2f} s")
    print(f"  Overshoot          : {m['overshoot_pct']:.1f} %")
    print(f"  Settling time (5%) : {m['settling_time']:.2f} s")
    print(f"  Steady-state error : {m['steady_state_error']:+.3f} m")
    print(f"  Final altitude     : {m['final_value']:.3f} m")
    print("="*44)

report(demo)


## 4. 🎬 The capstone animation

An animated drone with a live altitude trace and a live thrust gauge. This is your simulator flying.


In [ ]:
res = simulate(PIDController(Kp=4.0, Ki=1.2, Kd=4.0), total_time=14.0, max_thrust=25.0)
t, x, thr = res["t"], res["x"], res["thrust"]
fids = np.linspace(0, len(t)-1, 130).astype(int)

fig = plt.figure(figsize=(12, 6))
axd = fig.add_subplot(1, 3, 1); axt = fig.add_subplot(1, 3, 2); axg = fig.add_subplot(1, 3, 3)
# drone
axd.set_xlim(-1, 1); axd.set_ylim(0, 13); axd.set_xticks([]); axd.set_ylabel("altitude (m)")
axd.set_title("drone"); axd.axhline(0, color="saddlebrown", lw=3); axd.axhline(10, color="seagreen", ls="--", lw=2)
drone, = axd.plot([], [], "o", color="tab:blue", markersize=28)
# live altitude trace
axt.set_xlim(0, t[-1]); axt.set_ylim(0, 13); axt.axhline(10, color="seagreen", ls="--")
axt.set_title("altitude"); axt.set_xlabel("s"); axt.grid(True, alpha=0.3)
trace, = axt.plot([], [], color="tab:blue", lw=2)
# thrust gauge (bar)
axg.set_xlim(0, 1); axg.set_ylim(0, 26); axg.set_xticks([]); axg.set_title("thrust (N)")
axg.axhline(9.8, color="gray", ls=":", label="hover"); axg.legend(fontsize=8)
bar = axg.bar([0.5], [0], width=0.5, color="tab:purple")[0]
txt = axd.text(-0.9, 12, "", fontsize=10)

def init():
    drone.set_data([], []); trace.set_data([], []); bar.set_height(0); txt.set_text("")
    return drone, trace, bar, txt
def update(f):
    i = fids[f]; drone.set_data([0], [x[i]]); trace.set_data(t[:i+1], x[:i+1])
    bar.set_height(thr[i]); txt.set_text(f"t={t[i]:.1f}s\nx={x[i]:.2f} m")
    return drone, trace, bar, txt
ani = animation.FuncAnimation(fig, update, frames=len(fids), init_func=init, blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 5. 🎛️ The full cockpit

Every gain and every disturbance, live, with the dashboard and metrics updating on each change. (For
speed, this redraws the **static** dashboard rather than a fresh animation each time — use the cell
above whenever you want to *watch* a specific tuning fly.)


In [ ]:
def cockpit(Kp=4.0, Ki=1.2, Kd=4.0, target=10.0,
            noise=0.02, wind=0.0, delay_steps=0, max_thrust=25.0):
    ctrl = PIDController(Kp, Ki, Kd, d_filter=0.1)
    res = simulate(ctrl, target=target, noise_std=noise, wind=wind,
                   delay_steps=int(delay_steps), max_thrust=max_thrust, total_time=18.0)
    m = performance_metrics(res["t"], res["x"], target)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    ax[0].plot(res["t"], res["x"], color="tab:blue", lw=2)
    ax[0].plot(res["t"], res["target"], color="seagreen", ls="--")
    ax[0].set_ylim(0, max(target*1.5, 6)); ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("altitude (m)")
    ax[0].set_title(f"overshoot={m['overshoot_pct']:.0f}%  settle={m['settling_time']:.1f}s  "
                    f"sse={m['steady_state_error']:+.2f}m"); ax[0].grid(True, alpha=0.3)
    ax[1].plot(res["t"], res["thrust"], color="tab:purple", lw=1.4)
    ax[1].axhline(max_thrust, color="tab:red", ls=":"); ax[1].axhline(9.8, color="gray", ls=":")
    ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("thrust (N)"); ax[1].set_title("thrust command")
    ax[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(cockpit,
         Kp=FloatSlider(min=0.0, max=10.0, step=0.5, value=4.0),
         Ki=FloatSlider(min=0.0, max=4.0,  step=0.2, value=1.2),
         Kd=FloatSlider(min=0.0, max=10.0, step=0.5, value=4.0),
         target=FloatSlider(min=4.0, max=14.0, step=1.0, value=10.0),
         noise=FloatSlider(min=0.0, max=0.1, step=0.01, value=0.02),
         wind=FloatSlider(min=-6.0, max=6.0, step=0.5, value=0.0),
         delay_steps=IntSlider(min=0, max=25, step=1, value=0),
         max_thrust=FloatSlider(min=12.0, max=30.0, step=1.0, value=25.0));


## 6. 🏆 Challenge problems

Time to fly solo. Each challenge sets up a scenario and asks you to find gains that meet a goal.
Try it yourself first (edit the gains, rerun), then open the solution. There's rarely one right
answer — anything that meets the goal is correct.


### Challenge 1 — Setpoint tracking 🎯

The target **jumps from 10 m to 6 m at t = 8 s** (a commanded descent). Find gains so the drone
tracks *both* levels with small overshoot and settles quickly at each. The scenario is set up for
you; tune `Kp, Ki, Kd`.


In [ ]:
def target_step(t):
    return 10.0 if t < 8.0 else 6.0        # commanded altitude changes mid-flight

# >>> YOUR TUNING HERE <<<
ctrl = PIDController(Kp=4.0, Ki=1.2, Kd=4.0, d_filter=0.1)
res = simulate(ctrl, target=target_step, total_time=16.0, max_thrust=25.0, noise_std=0.01)
dashboard(res, "Challenge 1: track 10 m -> 6 m", max_thrust=25.0)


<details><summary>▸ Solution & discussion</summary>

Gains around <code>Kp=5, Ki=1.5, Kd=5</code> track both levels crisply. The step *down* is easy —
gravity helps — so the main risk is undershooting past 6 m; a bit more Kd keeps the descent damped.
Because our derivative is taken on the **measurement**, the sudden target change does *not* cause a
derivative kick (that's the payoff from Notebook 06). The integral re-settles to a lower value after
the step, since less-than-hover thrust is needed while descending, then returns to ~m·g at 6 m.
</details>


### Challenge 2 — Payload drop 📦

The drone hovers at 10 m carrying a package, so its **mass is 1.5 kg**. At **t = 8 s it drops the
package** and mass suddenly falls to 1.0 kg — the drone lurches upward. Find gains that recover to
10 m quickly with minimal jump.


In [ ]:
def mass_drop(t):
    return 1.5 if t < 8.0 else 1.0        # package released at t=8s -> lighter drone

# >>> YOUR TUNING HERE <<<
ctrl = PIDController(Kp=5.0, Ki=1.5, Kd=5.0, d_filter=0.1)
res = simulate(ctrl, target=10.0, mass=mass_drop, total_time=16.0, max_thrust=30.0)
dashboard(res, "Challenge 2: payload drop at t=8s", max_thrust=30.0)
print(f"Peak altitude after drop: {res['x'][400:].max():.2f} m (target 10)")


<details><summary>▸ Solution & discussion</summary>

The moment mass drops, the same thrust produces more acceleration (<code>a=F/m</code> with smaller
m), so the drone jumps up — a classic **disturbance**. Strong <b>D</b> (e.g. <code>Kd≈6</code>)
limits the jump by braking the sudden upward velocity, while the <b>I</b> term re-learns the new
hover thrust (now ~9.8 N instead of ~14.7 N) to remove any leftover error. Gains like
<code>Kp=5, Ki=1.5, Kd=6</code> recover smartly. Note the integral had grown larger to hold the
heavier drone; after the drop it must *shrink*, which takes a moment — that's the small dip you see
as it re-settles.
</details>


### Challenge 3 — Gust rejection 💨

The drone hovers at 10 m. At **t = 6 s a 3-second downward gust** hits (an extra −8 N force), then
it passes. Find gains that keep the altitude deviation as small as possible during the gust and
recover fast afterward.


In [ ]:
def gust(t):
    return -8.0 if 6.0 <= t < 9.0 else 0.0    # a temporary downdraft

# >>> YOUR TUNING HERE <<<
ctrl = PIDController(Kp=6.0, Ki=1.5, Kd=6.0, d_filter=0.1)
res = simulate(ctrl, target=10.0, wind=gust, total_time=16.0, max_thrust=30.0)
dashboard(res, "Challenge 3: 3-second downward gust from t=6s", max_thrust=30.0)
print(f"Lowest altitude during gust: {res['x'][300:500].min():.2f} m (target 10)")


<details><summary>▸ Solution & discussion</summary>

Gust rejection rewards a **stiffer** controller: higher <b>Kp</b> pushes back hard the instant the
drone dips, and <b>D</b> resists the downward velocity. The <b>I</b> term helps during the sustained
part of the gust but is slower to react. Something like <code>Kp=8, Ki=1.5, Kd=7</code> keeps the
dip shallow. The trade-off: those stiffer gains are twitchier with noise/delay, so on real hardware
you'd balance gust rejection against the disturbances of Notebook 09.
</details>


### Challenge 4 — Minimal-overshoot climb 📈

The purist's challenge: climb from the ground to 10 m with **overshoot under 2%** and **settling
time under 6 s**, on a lightly noisy sensor. Beat both targets. The cell prints your metrics.


In [ ]:
# >>> YOUR TUNING HERE <<<
ctrl = PIDController(Kp=3.0, Ki=0.8, Kd=5.0, d_filter=0.1)
res = simulate(ctrl, target=10.0, total_time=16.0, noise_std=0.015, max_thrust=25.0)
m = performance_metrics(res["t"], res["x"], 10.0)
plt.figure(figsize=(9, 4.4)); plt.plot(res["t"], res["x"], color="tab:blue", lw=2)
plt.axhline(10, color="seagreen", ls="--"); plt.axhspan(9.8, 10.2, color="seagreen", alpha=0.12)
plt.xlabel("time (s)"); plt.ylabel("altitude (m)"); plt.grid(True, alpha=0.3)
plt.title(f"overshoot={m['overshoot_pct']:.2f}%  settle={m['settling_time']:.2f}s"); plt.show()
ok = m["overshoot_pct"] < 2.0 and m["settling_time"] < 6.0
print(("PASSED both targets!" if ok else "Not yet -- keep tuning.") +
      f"  (overshoot {m['overshoot_pct']:.2f}%, settle {m['settling_time']:.2f}s)")


<details><summary>▸ Solution & discussion</summary>

Low overshoot means favouring <b>damping over speed</b>: modest <code>Kp</code>, generous
<code>Kd</code>, and a smaller <code>Ki</code> so the integral doesn't push past target. Try
<code>Kp=3, Ki=0.8, Kd=6</code> (or nudge Kd up to ~7). The tension: too much Kd amplifies the
sensor noise, so keep <code>d_filter</code> around 0.1. This challenge captures the essence of
tuning — there is no free lunch, only a well-chosen compromise.
</details>


## 🎓 Course wrap-up

You started not knowing what a derivative was in this context, and you've finished building and
tuning a complete PID flight controller that survives noise, saturation, wind, delay, payload
changes, and gusts. That's the real thing — the same algorithm running in drones, cars, thermostats,
3D printers, and industrial plants worldwide.

**The one-paragraph summary of everything:**
> Measure the error. **P** pushes proportional to it (fast, but leaves a gap and can oscillate).
> **I** accumulates it over time (erases the gap, but can wind up). **D** reacts to its rate of
> change (damps and anticipates, but amplifies noise). Add the three, defend against the real world
> with anti-windup and derivative filtering, and *tune* the balance for the disturbances you face.

### Where to go next
- **2D / 3D flight:** give the drone x, y, z and multiple rotors — you'll run several PID loops at
  once (position → desired tilt → motor speeds), a structure called **cascaded control**.
- **Better tuning:** explore automatic tuners, gain scheduling, and frequency-domain methods.
- **Beyond PID:** LQR, model-predictive control (MPC), and learning-based control — but PID remains
  the baseline they're all measured against.
- **Real hardware:** try a cheap flight-controller board or a simulator like a physics sandbox, and
  watch every lesson from Notebook 09 come to life.

Thanks for flying through this course. You've earned your wings. 🚁🎓
